In [80]:
import pandas as pd
import numpy as np


In [81]:
np.random.seed(42)


In [82]:
TASK_TYPES = {
    "assignment": 1,
    "exam": 2,
    "reading": 3,
    "project": 4
}


In [83]:
PRIORITY_LABELS = {
    0: "Low Priority",
    1: "Tomorrow",
    2: "Today"
}


In [84]:
def calculate_priority_score(
    days_left,
    task_type,
    effort_hours,
    attendance_risk,
    consistency,
    overdue_count
):
    score = 0

    # Deadline pressure
    if days_left <= 1:
        score += 4
    elif days_left <= 3:
        score += 3
    elif days_left <= 6:
        score += 2
    else:
        score += 1

    # Task type importance
    if task_type == 4:        # Project
        score += 4
    elif task_type == 2:      # Exam
        score += 3
    elif task_type == 1:      # Assignment
        score += 2
    else:                     # Reading
        score += 1

    # Effort
    if effort_hours >= 7:
        score += 3
    elif effort_hours >= 4:
        score += 2
    else:
        score += 1

    # Course credit impact (scaled)
    if course_credit >= 8:
       score += 3
    elif course_credit >= 5:
       score += 2
    elif course_credit >= 2:
       score += 1


    # Attendance risk
    score += attendance_risk * 2

    # Low consistency penalty
    if consistency < 0.6:
        score += 2

    # Overdue penalty
    score += overdue_count * 2

    # ✅ Small randomness to simulate human inconsistency
    score += np.random.choice([-1, 0, 0, 1])

    return score


In [85]:
def score_to_label(score):
    if score >= 14:
        return 2   # Today
    elif score >= 9:
        return 1   # Tomorrow
    else:
        return 0   # Low Priority


In [86]:
rows = []

NUM_SAMPLES = 80000  # Enough for strong learning

for _ in range(NUM_SAMPLES):
    days_left = np.random.randint(0, 15)
    task_type = np.random.choice(list(TASK_TYPES.values()))
    effort_hours = np.random.randint(1, 10)
    course_credit = np.random.randint(0, 11)
    attendance_risk = np.random.choice([0, 1, 2])  # Safe, Risk, Critical
    consistency = round(np.random.uniform(0.4, 1.0), 2)
    overdue_count = np.random.randint(0, 3)

    score = calculate_priority_score(
        days_left,
        task_type,
        effort_hours,
        attendance_risk,
        consistency,
        overdue_count
    )

    label = score_to_label(score)

    rows.append([
        days_left,
        task_type,
        effort_hours,
        course_credit,
        attendance_risk,
        consistency,
        overdue_count,
        label
    ])


In [87]:
columns = [
    "days_left",
    "task_type",
    "effort_hours",
    "course_credit",
    "attendance_risk",
    "consistency",
    "overdue_count",
    "label"
]

df = pd.DataFrame(rows, columns=columns)
df.head()


,days_left,task_type,effort_hours,course_credit,attendance_risk,consistency,overdue_count,label
0,6,4,8,4,2,0.67,2,2
1,10,4,5,3,2,0.41,1,2
2,11,2,6,1,0,0.77,1,0
3,12,4,9,0,2,0.77,1,2
4,11,4,3,6,0,0.43,2,2


In [88]:
df["label"].value_counts(normalize=True)


label
1    0.495300
2    0.397662
0    0.107037
Name: proportion, dtype: float64

In [89]:
df.to_csv("acadbox_task_priority_data.csv", index=False)
print("Dataset saved successfully.")


Dataset saved successfully.


In [90]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import joblib


In [91]:
df = pd.read_csv("acadbox_task_priority_data.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (80000, 8)


,days_left,task_type,effort_hours,course_credit,attendance_risk,consistency,overdue_count,label
0,6,4,8,4,2,0.67,2,2
1,10,4,5,3,2,0.41,1,2
2,11,2,6,1,0,0.77,1,0
3,12,4,9,0,2,0.77,1,2
4,11,4,3,6,0,0.43,2,2


In [92]:
X = df.drop("label", axis=1)
y = df["label"]

print("Features:", list(X.columns))


Features: ['days_left', 'task_type', 'effort_hours', 'course_credit', 'attendance_risk', 'consistency', 'overdue_count']


In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)


In [94]:
model = RandomForestClassifier(
    n_estimators=400,
    max_depth=16,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)



In [95]:
model.fit(X_train, y_train)
print("Model training completed.")


Model training completed.


In [96]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 90.07 %

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.88      0.85      1284
           1       0.91      0.89      0.90      5944
           2       0.91      0.92      0.92      4772

    accuracy                           0.90     12000
   macro avg       0.88      0.90      0.89     12000
weighted avg       0.90      0.90      0.90     12000



In [97]:
import joblib

joblib.dump(model, "acadbox_task_priority_model.pkl")
print("Model saved successfully")


Model saved successfully


In [98]:
model = joblib.load("acadbox_task_priority_model.pkl")


In [113]:
print("Enter task details:")

days_left = int(input("Days left to deadline: "))
task_type = int(input("Task type (1=Assignment, 2=Exam, 3=Reading, 4=Project): "))
effort_hours = int(input("Effort hours: "))
course_credit = int(input("Course credit: "))
attendance_risk = int(input("Attendance risk (0=Safe, 1=Risk, 2=Critical): "))
consistency = float(input("Consistency (0–1): "))
overdue_count = int(input("Overdue count: "))


Enter task details:


Days left to deadline:  0
Task type (1=Assignment, 2=Exam, 3=Reading, 4=Project):  2
Effort hours:  4
Course credit:  5
Attendance risk (0=Safe, 1=Risk, 2=Critical):  0
Consistency (0–1):  0.3
Overdue count:  0


In [114]:
features = [[
    days_left,
    task_type,
    effort_hours,
    course_credit,
    attendance_risk,
    consistency,
    overdue_count
]]

prediction = model.predict(features)[0]

LABEL_MAP = {
    0: "Low Priority",
    1: "Tomorrow",
    2: "Today"
}

print("Predicted Priority:", LABEL_MAP[prediction])


Predicted Priority: Tomorrow


/home/punit/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
